In [111]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)

df = pd.read_csv(r'C:\Users\Kirno\Downloads\verwerkte_data_finalversion.csv', sep=';')

uitsluiten = ["Wijk",
              "mvc Bloeddruk (Bovendruk) 1", "mvc Bloeddruk (Bovendruk) 2", 
              "mvc Cholesterol 1", "mvc Cholesterol 2", 
              "mvc Non-HDL 1", "mvc Non-HDL 2", 
              "mvc Bloedsuiker 1", "mvc Bloedsuiker 2", 
              "mvc BMI 1", "mvc BMI 2",
              "HR Diff"]

kolommen = [c for c in df.columns if c not in uitsluiten]

nieuwe_kolommen = {}

for kolom in kolommen:
    origineel = df[kolom].to_numpy()

    nieuwe_kolommen[f"{kolom}_man"] = np.clip(
        origineel + rng.uniform(-0.2, 0.2, len(df)), 0, None
    )

    nieuwe_kolommen[f"{kolom}_vrouw"] = np.clip(
        origineel + rng.uniform(-0.2, 0.2, len(df)), 0, None
    )

    nieuwe_kolommen[f"{kolom}_anders"] = np.clip(
        origineel + rng.uniform(-0.2, 0.2, len(df)), 0, None
    )

df = df.drop(columns=kolommen)

# alles in één keer toevoegen
df = pd.concat([df, pd.DataFrame(nieuwe_kolommen)], axis=1)
    
    
origineel = pd.to_numeric(df["HR Diff"], errors="coerce").to_numpy()

df["HR Diff_man"] = origineel + rng.uniform(-0.2, 0.2, len(df))
df["HR Diff_vrouw"] = origineel + rng.uniform(-0.2, 0.2, len(df))
df["HR Diff_anders"] = origineel + rng.uniform(-0.2, 0.2, len(df))

df.drop(columns=["HR Diff"], inplace=True)
    

df_copy = df.copy()


In [113]:

with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    print(df_copy)

                                     Wijk mvc Bloeddruk (Bovendruk) 1  \
0                         Bijlmer-Centrum                        <120   
1                            Bijlmer-Oost                        <120   
2                            Bijlmer-West                     140-180   
3                           Bos en Lommer                        <120   
4                   Buitenveldert, Zuidas                     120-140   
5                            Centrum-Oost                        <120   
6                            Centrum-West                     120-140   
7           De Aker, Sloten, Nieuw-Sloten                        <120   
8                  De Pijp, Rivierenbuurt                        <120   
9                              Gaasperdam                     120-140   
10                 Geuzenveld, Slotermeer                        <120   
11                IJburg, Zeeburgereiland                        <120   
12  Indische Buurt, Oostelijk Havengebied          

In [115]:
import random

config = {
    "mvc Bloeddruk (Bovendruk) 1": (
        ["<120", "120-140", "140-180", ">180"],
        [45, 35, 15, 5]
    ),
    "mvc Bloeddruk (Bovendruk) 2": (
        ["<120", "120-140", "140-180", ">180"],
        [45, 35, 15, 5]
    ),
    "mvc Cholesterol 1": (
        ["<5", "5-6.5", "6.5-8", ">8"],
        [65, 24, 8, 3]
    ),
    "mvc Cholesterol 2": (
        ["<5", "5-6.5", "6.5-8", ">8"],
        [65, 24, 8, 3]
    ),
    "mvc Non-HDL 1": (
        ["<3.8", ">3.8"],
        [76, 24]
    ),
    "mvc Non-HDL 2": (
        ["<3.8", ">3.8"],
        [76, 24]
    ),
    "mvc Bloedsuiker 1": (
        ["<7.8", "7.8-11", ">11.1"],
        [73, 23, 4]
    ),
    "mvc Bloedsuiker 2": (
        ["<7.8", "7.8-11", ">11.1"],
        [73, 23, 4]
    ),
    "mvc BMI 1": (
        ["<18.5", "18.5-25", "25-30", ">30"],
        [10, 60, 25, 5]
    ),
    "mvc BMI 2": (
        ["<18.5", "18.5-25", "25-30", ">30"],
        [10, 60, 25, 5]
    )
}

result = df_copy.copy()

for kolom, (categories, weights) in config.items():
    for groep in ["man", "vrouw", "anders"]:
        result[f"{kolom}_{groep}"] = random.choices(
            categories,
            weights=weights,
            k=len(result)
        )

In [117]:
mvc = ["mvc Bloeddruk (Bovendruk) 1", "mvc Bloeddruk (Bovendruk) 2", "mvc Cholesterol 1", "mvc Cholesterol 2", "mvc Non-HDL 1", "mvc Non-HDL 2", "mvc Bloedsuiker 1", "mvc Bloedsuiker 2", "mvc BMI 1", "mvc BMI 2"]
result = result.drop(columns=mvc)

print(result)

                                     Wijk    sdnn_1_man  sdnn_1_vrouw  \
0                         Bijlmer-Centrum     47.969582     47.737855   
1                            Bijlmer-Oost     29.715551     29.726688   
2                            Bijlmer-West    868.363439    868.037522   
3                           Bos en Lommer    375.868947    375.651716   
4                   Buitenveldert, Zuidas     50.237671     50.473220   
5                            Centrum-Oost    126.970249    126.877905   
6                            Centrum-West     25.794456     25.877004   
7           De Aker, Sloten, Nieuw-Sloten     87.844426     87.660330   
8                  De Pijp, Rivierenbuurt     27.281245     27.378184   
9                              Gaasperdam   1626.450154   1626.457822   
10                 Geuzenveld, Slotermeer   1050.868319   1050.795789   
11                IJburg, Zeeburgereiland   8500.490706   8500.171969   
12  Indische Buurt, Oostelijk Havengebied  10737.54

In [119]:
import numpy as np
import pandas as pd

df_uitgebreid = (
    result.loc[result.index.repeat(3)]
    .reset_index(drop=True)
)

df_uitgebreid["leeftijdscategorie"] = (
    ["40-50", "50-60", "60-70"] * len(result)
)

kolommen = [
    col for col in df_uitgebreid.columns
    if (
        pd.api.types.is_numeric_dtype(df_uitgebreid[col])
        and "mvc" not in col.lower()
        and "diff" not in col.lower()
        and col != "Wijk"
        and col != "leeftijdscategorie"
    )
]

for kolom in kolommen:
    factor = np.random.uniform(0.95, 1.1, size=len(df_uitgebreid))

    df_uitgebreid[kolom] = (
        df_uitgebreid[kolom].astype(float) * factor
    ).round(2)

print(df_uitgebreid)

               Wijk  sdnn_1_man  sdnn_1_vrouw  sdnn_1_anders  rmssd_1_man  \
0   Bijlmer-Centrum       47.80         47.94          46.72        81.74   
1   Bijlmer-Centrum       46.56         51.69          50.33        86.64   
2   Bijlmer-Centrum       50.32         51.76          50.55        79.23   
3      Bijlmer-Oost       30.82         28.28          30.14        15.34   
4      Bijlmer-Oost       30.49         28.28          29.78        15.85   
..              ...         ...           ...            ...          ...   
70  Weesp, Driemond      125.43        116.13         111.41       126.43   
71  Weesp, Driemond      112.21        115.53         117.65       125.09   
72       Westerpark       57.93         64.69          58.28        46.93   
73       Westerpark       64.37         57.64          60.58        51.63   
74       Westerpark       57.91         63.61          63.17        46.95   

    rmssd_1_vrouw  rmssd_1_anders  nn50_1_man  nn50_1_vrouw  nn50_1_anders 

In [121]:
config = {
    "mvc Bloeddruk (Bovendruk) 1": (
        ["<120", "120-140", "140-180", ">180"],
        [45, 35, 15, 5]
    ),
    "mvc Bloeddruk (Bovendruk) 2": (
        ["<120", "120-140", "140-180", ">180"],
        [45, 35, 15, 5]
    ),
    "mvc Cholesterol 1": (
        ["<5", "5-6.5", "6.5-8", ">8"],
        [65, 24, 8, 3]
    ),
    "mvc Cholesterol 2": (
        ["<5", "5-6.5", "6.5-8", ">8"],
        [65, 24, 8, 3]
    ),
    "mvc Non-HDL 1": (
        ["<3.8", ">3.8"],
        [76, 24]
    ),
    "mvc Non-HDL 2": (
        ["<3.8", ">3.8"],
        [76, 24]
    ),
    "mvc Bloedsuiker 1": (
        ["<7.8", "7.8-11", ">11.1"],
        [73, 23, 4]
    ),
    "mvc Bloedsuiker 2": (
        ["<7.8", "7.8-11", ">11.1"],
        [73, 23, 4]
    ),
    "mvc BMI 1": (
        ["<18.5", "18.5-25", "25-30", ">30"],
        [10, 60, 25, 5]
    ),
    "mvc BMI 2": (
        ["<18.5", "18.5-25", "25-30", ">30"],
        [10, 60, 25, 5]
    )
}

age_shift = {
    "40-50": -1,   # iets gezonder
    "50-60": 0,
    "60-70": 1     # iets slechter
}

def shift_weights(weights, shift):
    weights = np.array(weights, dtype=float)

    if shift > 0:
        for _ in range(shift):
            weights = np.roll(weights, -1)  # schuift risico naar slechtere categorie
    elif shift < 0:
        for _ in range(-shift):
            weights = np.roll(weights, 1)   # schuift naar gezonder

    return weights


df_uitgebreid = df_uitgebreid.copy()

for kolom, (categories, weights) in config.items():
    for groep in ["man", "vrouw", "anders"]:

        resultaten = []

        for leeftijd in df_uitgebreid["leeftijdscategorie"]:
            w = shift_weights(weights, age_shift[leeftijd])

            resultaten.append(
                random.choices(categories, weights=w, k=1)[0]
            )

        df_uitgebreid[f"{kolom}_{groep}"] = resultaten


In [123]:
df_uitgebreid

,Wijk,sdnn_1_man,sdnn_1_vrouw,sdnn_1_anders,rmssd_1_man,rmssd_1_vrouw,rmssd_1_anders,nn50_1_man,nn50_1_vrouw,nn50_1_anders,...,mvc Bloedsuiker 2_man,mvc Bloedsuiker 2_vrouw,mvc Bloedsuiker 2_anders,mvc BMI 1_man,mvc BMI 1_vrouw,mvc BMI 1_anders,mvc BMI 2_man,mvc BMI 2_vrouw,mvc BMI 2_anders,leeftijdscategorie
0,Bijlmer-Centrum,47.80,47.94,46.72,81.74,85.86,86.38,5.59,4.83,5.46,...,7.8-11,7.8-11,7.8-11,25-30,25-30,25-30,25-30,>30,25-30,40-50
1,Bijlmer-Centrum,46.56,51.69,50.33,86.64,84.35,84.35,4.97,5.28,5.15,...,<7.8,<7.8,<7.8,18.5-25,25-30,18.5-25,>30,18.5-25,18.5-25,50-60
2,Bijlmer-Centrum,50.32,51.76,50.55,79.23,83.27,78.36,5.33,5.08,5.46,...,>11.1,>11.1,>11.1,<18.5,<18.5,>30,18.5-25,<18.5,<18.5,60-70
3,Bijlmer-Oost,30.82,28.28,30.14,15.34,15.67,15.29,1.27,0.95,1.20,...,7.8-11,7.8-11,7.8-11,25-30,25-30,>30,18.5-25,18.5-25,18.5-25,40-50
4,Bijlmer-Oost,30.49,28.28,29.78,15.85,16.21,15.74,1.17,0.93,1.30,...,7.8-11,<7.8,<7.8,25-30,25-30,18.5-25,18.5-25,18.5-25,18.5-25,50-60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,"Weesp, Driemond",125.43,116.13,111.41,126.43,128.80,138.30,68.35,66.90,65.01,...,<7.8,<7.8,<7.8,18.5-25,<18.5,18.5-25,18.5-25,<18.5,25-30,50-60
71,"Weesp, Driemond",112.21,115.53,117.65,125.09,124.54,125.24,67.13,69.43,70.32,...,>11.1,>11.1,<7.8,<18.5,>30,<18.5,<18.5,<18.5,>30,60-70
72,Westerpark,57.93,64.69,58.28,46.93,48.77,49.04,37.49,40.95,38.33,...,7.8-11,<7.8,>11.1,25-30,<18.5,>30,25-30,<18.5,>30,40-50
73,Westerpark,64.37,57.64,60.58,51.63,52.47,45.90,38.19,41.13,38.59,...,<7.8,7.8-11,7.8-11,18.5-25,25-30,18.5-25,25-30,25-30,25-30,50-60


In [125]:
kolommen = [
    "HR Diff_man",
    "HR Diff_vrouw",
    "HR Diff_anders"
]

factor = np.random.uniform(0.95, 1.1, size=len(df_uitgebreid))

for kolom in kolommen:
    df_uitgebreid[kolom] = (
        pd.to_numeric(df_uitgebreid[kolom], errors="coerce") * factor
    ).round(2)

In [127]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    print(df_uitgebreid)

                                     Wijk  sdnn_1_man  sdnn_1_vrouw  \
0                         Bijlmer-Centrum       47.80         47.94   
1                         Bijlmer-Centrum       46.56         51.69   
2                         Bijlmer-Centrum       50.32         51.76   
3                            Bijlmer-Oost       30.82         28.28   
4                            Bijlmer-Oost       30.49         28.28   
5                            Bijlmer-Oost       31.00         32.69   
6                            Bijlmer-West      932.40        838.38   
7                            Bijlmer-West      944.03        884.76   
8                            Bijlmer-West      848.53        896.42   
9                           Bos en Lommer      390.31        383.12   
10                          Bos en Lommer      411.34        399.09   
11                          Bos en Lommer      386.08        376.18   
12                  Buitenveldert, Zuidas       48.12         54.25   
13    

In [129]:
df_uitgebreid.shape

(75, 107)

In [133]:
df_uitgebreid.to_csv(r'C:\Users\Kirno\Streamlit\verwerkte_data_geslacht_leeftijd_final.csv', index=False)